In [1]:
import pandas as pd
import duckdb
import dlt
import numpy as np

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Read CSV 
path = "../data/sample.csv"
def read_data(path):
    df = pd.read_csv(path)
    return df

df = read_data(path) 
df.head()

,order_id,product,category,region,quantity,price,revenue
0,1.0,Laptop,Electronics,North,2,50000,100000
1,2.0,Phone,Electronics,South,3,20000,60000
2,3.0,Desk,Furniture,West,1,10000,10000
3,4.0,Chair,Furniture,East,4,3000,12000
4,5.0,Monitor,Electronics,North,2,15000,30000


In [6]:
def transform_data(df):
    #to remove entries with null values 
    df["remarks"]=np.where(df["order_id"].isna(),"invalid","valid")

    removed = df[df["remarks"]=="invalid"]
    df = df[df["remarks"]=="valid"]
    
    df["order_id"]=df["order_id"].astype("int")
    df["order_id"]=df["order_id"].astype("string")

    columns = ["quantity","price","revenue"]

    for col in columns :
        df[col]=df[col].astype("float")
        df[col]=df[col].abs()

    columns = ["product","category","region"]

    for col in columns :
        df[col]=df[col].fillna("Unknown")

    print("Data transformations completed successfully!")
    
    print(df)
    print(removed)

    return df
    
cleaned_data=transform_data(df)
cleaned_data = cleaned_data.to_dict(orient="records")

Data transformations completed successfully!
  order_id   product     category   region  quantity    price   revenue  \
0        1    Laptop  Electronics    North       2.0  50000.0  100000.0   
1        2     Phone  Electronics    South       3.0  20000.0   60000.0   
2        3      Desk    Furniture     West       1.0  10000.0   10000.0   
3        4     Chair    Furniture     East       4.0   3000.0   12000.0   
4        5   Monitor  Electronics    North       2.0  15000.0   30000.0   
5        6    Tablet  Electronics    South       1.0  25000.0   25000.0   
6        7      Sofa    Furniture     West       1.0  40000.0   40000.0   
7        8  Keyboard  Electronics  Unknown       5.0   1000.0    5000.0   
8        9     Mouse  Electronics    North      10.0    500.0    5000.0   
9       10   Printer  Electronics    South       2.0   8000.0   16000.0   

  remarks  
0   valid  
1   valid  
2   valid  
3   valid  
4   valid  
5   valid  
6   valid  
7   valid  
8   valid  
9   valid

In [17]:
#isse humari ek data pipeline ban jaegi --> 
def load_data_db(cleaned_data):
    data_load_pipeline = dlt.pipeline(
    pipeline_name = "data_load_pipeline",
    destination = "duckdb", #ye aapka db type hoga :-> postgres/duckdb/snowflake/bigquery
    dataset_name = "sales_data"
    )

    print("Pipeline Created")

    info=data_load_pipeline.run(
    cleaned_data, 
    table_name = "sales",
    write_disposition = "replace"
    )

    print("Data Pushed")

    #lets check if data is pushed

    db_path = data_load_pipeline.sql_client().credentials.database
    con = duckdb.connect(db_path)
    loaded_table=con.execute("Select * from sales_data.sales;").fetchdf()

    print(loaded_table.head())
    
    return  
load_data_db(cleaned_data)

Pipeline Created
Data Pushed
  order_id  product     category region  quantity    price   revenue remarks  \
0        1   Laptop  Electronics  North       2.0  50000.0  100000.0   valid   
1        2    Phone  Electronics  South       3.0  20000.0   60000.0   valid   
2        3     Desk    Furniture   West       1.0  10000.0   10000.0   valid   
3        4    Chair    Furniture   East       4.0   3000.0   12000.0   valid   
4        5  Monitor  Electronics  North       2.0  15000.0   30000.0   valid   

        _dlt_load_id         _dlt_id  
0  1781011689.818902  imP1Ji59KEW9OA  
1  1781011689.818902  5arG91C4lhQeMQ  
2  1781011689.818902  LGATYuo/2tkkag  
3  1781011689.818902  oKSbDdjSifC5ag  
4  1781011689.818902  N6OlhyJ/Tkrtiw  


   version  engine_version       pipeline_name  \
0        1               4  data_load_pipeline   
1        2               4  data_load_pipeline   

                                               state  \
0  eNpdj8uKwkAQRf+l1kEYRkECs5HgQtCZMItBBykqSZlubD...   
1  eNpdj1FLw0AQhP/Lvhr6IDFIwBeRWigqKYQKRZb17mqOXC...   

                        created_at  \
0 2026-06-09 08:05:00.154716+00:00   
1 2026-06-09 13:20:32.675454+00:00   

                                   version_hash        _dlt_load_id  \
0  FB38lyD5Sb8iPsQZsqbJzkhFoDMfg4OpgzDNWechoYQ=  1780992300.1358032   
1  ckQydQB6Wk4i8Rs+MHU64X6xGvtK5FFQchOfUNKDRtU=  1781011232.6578517   

          _dlt_id  
0  FbX5LhUJpx3mNg  
1  Fui5ah4bUdwgiQ  


In [9]:
print(info

Pipeline data_load_pipeline load step completed in 0.18 seconds
1 load package(s) were loaded to destination duckdb and into dataset sales_data
The duckdb destination used duckdb:////workspaces/Local-Data-Engineering-Environment-with-dlt-DuckDB-Jupyter/notebooks/data_load_pipeline.duckdb location to store data
Load package 1781011232.6578517 is LOADED and contains no failed jobs


/workspaces/Local-Data-Engineering-Environment-with-dlt-DuckDB-Jupyter/notebooks/data_load_pipeline.duckdb


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  order_id   product     category   region  quantity    price   revenue  \
0        1    Laptop  Electronics    North       2.0  50000.0  100000.0   
1        2     Phone  Electronics    South       3.0  20000.0   60000.0   
2        3      Desk    Furniture     West       1.0  10000.0   10000.0   
3        4     Chair    Furniture     East       4.0   3000.0   12000.0   
4        5   Monitor  Electronics    North       2.0  15000.0   30000.0   
5        6    Tablet  Electronics    South       1.0  25000.0   25000.0   
6        7      Sofa    Furniture     West       1.0  40000.0   40000.0   
7        8  Keyboard  Electronics  Unknown       5.0   1000.0    5000.0   
8        9     Mouse  Electronics    North      10.0    500.0    5000.0   
9       10   Printer  Electronics    South       2.0   8000.0   16000.0   

  remarks        _dlt_load_id         _dlt_id  
0   valid  1781011232.6578517  iNcSXlJIFWUbhg  
1   valid  1781011232.6578517  A4Cg52NIveQ/4A  
2   valid  1781011232.6578517 